# 04 - Bayesian A/B Analysis with PyMC
**Milestone 5**

Instead of a p-value we report the **probability that B beats A**, the expected
loss from shipping the wrong variant, and a credible interval on the lift.
Binary metrics use a Beta-Binomial model; continuous metrics use a Normal model.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src")) and os.path.exists(
            os.path.join(d, "src", "config.py")
        ):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start)
        d = parent

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import artifact_path, raw_path, ensure_directories
ensure_directories()
FIGS = artifact_path("figures")
os.makedirs(FIGS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIGURES    :", FIGS)


## 1. Beta-Binomial model (PyMC) on a conversion experiment

In [ ]:
from src.core import bayesian as bayes
from src.core.conclude import bayesian_conclusion

res = bayes.bayesian_ab_binary(
    control_success=520, control_n=5000,
    treatment_success=575, treatment_n=5000,
    draws=2000, seed=42,
)
res["engine"], res["prob_treatment_better"], res["credible_interval"], res["decision"]

In [ ]:
bayesian_conclusion(res)

### Posterior distribution of the lift (B - A)

In [ ]:
hist = res["posterior_histogram"]
edges = np.array(hist["edges"]); counts = np.array(hist["counts"])
centers = (edges[:-1] + edges[1:]) / 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(centers, counts, width=(edges[1]-edges[0]), color="#4f46e5", alpha=0.85)
ax.axvline(0, color="#dc2626", ls="--", label="no effect")
ax.set_xlabel("lift  (B - A)"); ax.set_ylabel("posterior draws")
ax.set_title(f"P(B beats A) = {res['prob_treatment_better']:.1%}"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIGS, "04_posterior.png"), dpi=120, bbox_inches="tight"); plt.show()

## 2. How P(B beats A) grows with evidence

In [ ]:
ns = [200, 500, 1000, 2500, 5000]
probs = []
for n in ns:
    r = bayes.bayesian_ab_binary(int(0.104 * n), n, int(0.115 * n), n, draws=1500, seed=1)
    probs.append(r["prob_treatment_better"])
pd.DataFrame({"n_per_arm": ns, "P(B beats A)": [round(p, 3) for p in probs]})

## 3. Continuous metric - Normal-Normal model

In [ ]:
from src.data_pipeline.simulate import simulate_continuous_ab

dfc = simulate_continuous_ab(n_a=3000, n_b=3000, mean_a=50, relative_lift=0.04, std=15, seed=5)
c = dfc.loc[dfc.variant == "A", "revenue"].values
t = dfc.loc[dfc.variant == "B", "revenue"].values
res_c = bayes.bayesian_ab_continuous(c, t, draws=2000, seed=7)
{ "prob_B_better": round(res_c["prob_treatment_better"], 3),
  "mean_lift": round(res_c["posterior_mean_diff"], 3),
  "relative_lift": round(res_c["relative_lift_posterior_mean"], 4),
  "decision": res_c["decision"] }

## Takeaways
* Bayesian reporting answers the question stakeholders actually ask: "how likely is B better?"
* The expected-loss columns translate uncertainty into a concrete decision cost.
* Results agree closely with the frequentist analysis, but the interpretation does not depend on a hypothetical repeat-sampling argument.